# Build `ai_product` — bảng gold product-variant đã enrich

Hợp nhất **11 bảng Partner-Portal đã được normalize** trong `lh_vv_bronze.partner.*` thành một bảng denormalized duy nhất là `ai_product`, với **một dòng cho mỗi product** (`product_id`) và các variant được lồng trong `variants_json`.

Bảng này tuân theo Product contract của knowledge-base, phục vụ search hydration và `ProductDetail`.

**Hai product domain, một contract**

- **Holiday products** (`product_type="other"`) — được xây dựng từ 11 bảng holiday (`products` / `product_variants` / `product_posts` / …). Mỗi `product_id` là một product; các `product_variants` của product đó là các variant. Luồng điều hướng: POI (`business_service`) → product → variant.
- **Hotel products** (`product_type="hotel"`, `ENABLE_HOTEL`) — được xây dựng từ `hotel_room_types` và `hotel_rate_plans`. **Một product cho mỗi room type**; rate plan của hotel được lồng làm variant. Luồng điều hướng tương tự: POI (hotel) → product (room) → variant (rate plan). Cả hai bảng hotel dùng `business_service_id` làm key, nên hotel product được resolve POI giống holiday product và union vào cùng grain trước khi write. Các bảng nguồn này không có price, nên các price column của hotel là NULL.

**Dữ liệu được gộp vào mỗi holiday row**

| Nhóm | Columns |
|---|---|
| Variant | `variant_id, product_id, variant_status, variant_external_id, variant_attributes_json, variant_metadata_json, variant_created_at, variant_updated_at, variant_deleted_at, is_deleted_variant` |
| Variant name (vi) | `variant_name_vi, variant_description_vi` |
| Resolved attributes | `attributes_en` (map `en_label -> vi_value`), `attributes_labeled` (array `{attr_code, attr_name_vi, attr_name_en, value_code, value_name_vi}`) |
| Product | `product_line, product_status, product_effective_status, is_deleted_product, is_active, product_external_id, product_attributes_json, product_metadata_json, main_attribute_code, product_start_date, product_end_date, product_name_vi, product_name_en, product_description_vi` |
| Product rollups | `variant_count, active_variant_count`, các axis array: `variant_tickets / variant_dates / variant_hotels / variant_age_groups / variant_room_types / variant_departures / variant_statuses` |
| Post (pricing/content) | `product_post_id, product_post_internal_id, post_status, is_deleted_post, original_price, marketing_price, markup_percent, price_level, thumbnail_url, gallery_json, validity_from, validity_to, post_title_vi, post_short_description_vi, post_short_description_en, post_tags_json, post_ai_tags_json, post_sections_json, post_faqs_json` |
| POI mapping | `business_service_id` (scalar), `poi_brand`, `source_id`, `is_bookable` |

**Ý nghĩa của `is_active` và `is_bookable`**

```text
is_active =
  product_status là active
  AND product không bị soft-delete
```

`is_active` thể hiện product còn sống trong catalog.

```text
is_bookable =
  is_active
  AND use_date chưa hết hạn
  AND có ít nhất một active, non-deleted variant
  AND với DIFF: linked product_post active và không bị deleted
```

`is_bookable` thể hiện product hiện có thể được offer để booking. Vì vậy product có thể `is_active=true` nhưng `is_bookable=false`, ví dụ khi toàn bộ variant đã inactive hoặc DIFF post không còn active.

**Hotel row mapping** — `product_id` là room ID, `name` là room name (vi→en), và `variants_json` là các rate plan của hotel (`{meal_plan, kind, valid_from/to, include_tax, include_service_charge}`). Room chính là product, nên các thuộc tính riêng của room (`maxOccupancy, adultCapacity, childCapacity, maxExtraBed, roomSize, viewType, amenities, availableRooms, totalRooms`) được đưa vào `product_attributes`, cùng với `shortDescription` và `tags`—không đưa vào variant axes.

Các `variant_room_types` và `variant_*` axis khác là **NULL cho hotel**: room type là product, không phải variant dimension. Rate plan được gắn ở cấp hotel, nên mọi room của cùng hotel nhận cùng rate-plan list. Đây là xấp xỉ được chấp nhận cho đến khi có room↔rate link hoặc rates table.

**Translation** — `product_name` và post `short_description` được translate vi→en bằng Azure OpenAI (GPT-4.1), vì vậy bảng xuất ra `product_name_en` và `post_short_description_en`. Variant name và attribute value vẫn giữ tiếng Việt theo search API contract.

**POI mapping** — product dùng `business_service_id` duy nhất từ relation table, hoặc fallback sang `product_posts.business_service_id`, làm gold POI `source_id`. Trong `TEST_MODE`, `FORCE_PRODUCT_SOURCE_ID` và `FORCE_PRODUCT_DESTINATION_ID` tạm thời pin mọi product vào một smoke-test POI. `address`, `website_url`, và `general_amenities` được ingestion hydrate từ gold POI table.

Output là `lh_vv_gold.dbo.ai_product` (source: `lh_vv_bronze.partner.*`). `updated_at` là Event Hub sync watermark. Notebook có thể chạy lại; write sử dụng full `overwrite`.


In [1]:
# --- Parameters (override in a Fabric pipeline / "Parameters" cell) ---
BRONZE_LH   = "lh_vv_bronze"   # source lakehouse
SRC_SCHEMA  = "partner"        # source schema inside the lakehouse
LOCALE      = "vi"             # only locale present in source translations
TARGET_TABLE = "ai_product"    # output table name
TARGET_LH    = "lh_vv_gold.dbo"  # gold layer, dbo schema -> lh_vv_gold.dbo.ai_product
POI_GOLD_TABLE = "lh_vv_gold.dbo.ai_poi"
TRANSLATION_TABLE = "lh_vv_gold.dbo.ai_product_translation"  # per-product_id vi->en translation cache (skip already-translated product_ids)
ENRICHED_POI_TABLE = "lh_vv_gold.dbo.ai_enriched_poi"  # POI set ingested as KB POI nodes; a product's source_id must exist here (else it cannot resolve its POI link). Adjust path if the enriched POI table lives elsewhere.


# --- Bookability gates -------------------------------------------------------
# is_active is catalog lifecycle only: source status is active and not soft-deleted.
# REQUIRE_ACTIVE_POST_FOR_DIFF: a DIFF product without an active, non-deleted post
#   remains active but is_bookable=False. Non-DIFF products are unaffected.
# HIDE_EXPIRED_USE_DATE: an expired latest use_date leaves the product active but
#   sets is_bookable=False. Products with no use_date are unaffected.
REQUIRE_ACTIVE_POST_FOR_DIFF = False
HIDE_EXPIRED_USE_DATE        = True

# --- Azure OpenAI: vi -> en translation (mirrors the ingestion's GPT-4.1 step) ---
AZURE_AI_ENDPOINT    = "https://devagenticplatformapi.openai.azure.com"
AZURE_AI_DEPLOYMENT  = "gpt-4.1"
AZURE_AI_API_VERSION = "2024-12-01-preview"
AZURE_AI_API_KEY     = "REDACTED"      # <-- set before running (or pull from Key Vault / mssparkutils.credentials)
# vi->en GPT translation of product/room names. Applies to BOTH domains:
#   True  -> translate holiday product names AND hotel room names vi->en.
#   False -> skip translation entirely (no GPT calls); ship the RAW source text
#            AS the English value (so *_en may contain Vietnamese as-entered).
ENABLE_TRANSLATION   = False
# Translation throughput controls — batch many texts per GPT call and pause
# between calls so we do not trip Azure OpenAI rate limits (HTTP 429).
TRANSLATE_BATCH_SIZE      = 20    # vi strings per GPT request
TRANSLATE_DELAY_SEC       = 0.5   # pause between per-item fallback calls
TRANSLATE_BATCH_PAUSE_SEC = 3.0   # pause between batches
TRANSLATE_MAX_RETRIES     = 5     # retries (exp backoff) on 429 / 5xx

# --- Hotel inventory ingest --------------------------------------------------
# Fold the hotel partner tables into ai_product: one product per room type
# (hotel_room_types), with the hotel's rate plans (hotel_rate_plans) nested as
# variants. Both link to a gold POI by business_service_id (the hotel), so hotel
# products ride the same POI resolution / validation as holiday products below.
# These tables carry NO price -> hotel price columns stay NULL.
ENABLE_HOTEL = False            # False -> only holiday products are built

# --- CDC decimal decoding ---------------------------------------------------
# product_posts price columns arrive as Debezium/CDC Decimal: a big-endian
# two's-complement integer (binary, base64 when stringified) with a FIXED scale.
# Scales confirmed from the source decimal formatting (prod_data/product_posts.json):
#   prices  -> "1000000.0000"  = 4 decimals -> numeric(_,4) -> scale 4
#   markup  -> "20.00"         = 2 decimals -> numeric(_,2) -> scale 2
# e.g. unscaled 2,000,000,000 / 10^4 = 200,000.0000 VND.
# The decoder FALLS BACK to plain numeric values, so already-decimal columns
# pass through untouched.
PRICE_DECIMAL_SCALE  = 4
MARKUP_DECIMAL_SCALE = 2

# --- POI linkage -----------------------------------------------------------
# In production source_id comes from product_business_service_relation, falling
# back to product_posts.business_service_id. The source_id must resolve in ai_poi
# and ai_enriched_poi before the product is written.
#
# --- Smoke-test POI pin --------------------------------------------------------
# When set, EVERY product's source_id is forced to this value so all
# products link to one POI that the push notebook also ships in test mode.
# POI key across the pipeline is source_id (Google Place ID): ai_enriched_poi
# uses id_key=source_id, the POI graph node id = source_id, and products link by
# source_id. So pin to the first test POI's source_id. Set to None to disable.

TEST_MODE = False  # set False for production builds (disables the source_id pin below)
FORCE_PRODUCT_SOURCE_ID = "26edc517-c779-4fa9-ae8e-2e7f8589f9e5"  # smoke-test pin (applied ONLY when TEST)
FORCE_PRODUCT_DESTINATION_ID = "e8xuqhikk4547xjw4haffqko"  # Da Nang — destination of the pin POI (test mode); None -> use the gold-table join

DESTINATION_ID_REMAP = {
    'npl86zvhuytfb0ys679uz6w1': 'e8xuqhikk4547xjw4haffqko',  # Bana Hill
    'w9w1hdm8iwcofbh1rj12a3i2': 'chdm4mekx35eqiiscri33t30',  # Hue
    'bpy3b4acd6hjdisugue3sxz7': 'e8xuqhikk4547xjw4haffqko',  # Da Nang
    't36nuvkacqp87p6pk1k4zbv8': 'xjo4p6jhlvabpjhfyq4rgac7',  # Hoi An
}

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 3, Finished, Available, Finished, False)

In [2]:
import base64
import unicodedata
from pyspark.sql import functions as F, Window
from pyspark.sql.types import MapType, StringType, ArrayType


def _decode_cdc_decimal(value, scale):
    """Decode a price/percent value to float, tolerant of both encodings.

    Fallback order:
      1. already numeric (int/float)         -> use as-is
      2. plain decimal string ("1200000.0")  -> float()  (e.g. sample/JSON data)
      3. Debezium/CDC Decimal (raw bytes or base64 text) -> big-endian
         two's-complement integer / 10**scale

    So a column that's already a proper decimal passes through untouched; only
    the binary/base64 CDC form is scaled."""
    if value is None:
        return None
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return None
        try:
            return float(s)              # plain decimal string -> not CDC-encoded
        except ValueError:
            pass                         # fall through to base64/CDC decode
    try:
        raw = bytes(value) if isinstance(value, (bytes, bytearray)) else base64.b64decode(str(value))
        if not raw:
            return None
        return int.from_bytes(raw, "big", signed=True) / (10.0 ** scale)
    except Exception:
        return None


_price_udf = F.udf(lambda v: _decode_cdc_decimal(v, PRICE_DECIMAL_SCALE), "double")
_markup_udf = F.udf(lambda v: _decode_cdc_decimal(v, MARKUP_DECIMAL_SCALE), "double")


def decode_price(df, col):
    return _price_udf(F.col(col)) if col in df.columns else F.lit(None).cast("double")


def decode_markup(df, col):
    return _markup_udf(F.col(col)) if col in df.columns else F.lit(None).cast("double")


def _fold(s) -> str:
    """Lowercase + strip Vietnamese accents (đ->d) for keyword matching."""
    s = str(s if s is not None else "").lower()
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn").replace("đ", "d")


fold_udf = F.udf(_fold, StringType())

# Some bronze datetime columns hold out-of-range values (e.g. a micro/nanosecond
# epoch cast to TIMESTAMP -> year 289478). Spark 3's strict parser throws a
# SparkUpgradeException at the first action; LEGACY restores lenient parsing.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
# Two attribute codes can resolve to the same English label (e.g. both -> "hotel"),
# which makes attributes_en a map with a duplicate key. LAST_WIN matches the
# ingestion (its Python dict overwrites on repeat) instead of erroring.
spark.conf.set("spark.sql.mapKeyDedupPolicy", "LAST_WIN")


def read_src(table: str):
    """Read a bronze source table by 3-part name, falling back to spark.sql."""
    fq = f"{BRONZE_LH}.{SRC_SCHEMA}.{table}"
    try:
        return spark.read.table(fq)
    except Exception:
        return spark.sql(f"SELECT * FROM {fq}")


def optcol(df, col: str, dtype: str = "string"):
    """`col` if the bronze table has it, else a typed NULL literal.

    Bronze schemas drift from the JSON exports (e.g. product_variants may have
    no `metadata` column), so every non-key column is read defensively."""
    return F.col(col) if col in df.columns else F.lit(None).cast(dtype)


def _epoch_to_seconds(col):
    """Normalize a numeric epoch of unknown unit (s/ms/us/ns) to whole seconds.
    Bronze stores these inconsistently -- some are micro/nano epochs (casting
    straight to TIMESTAMP yields year 289478) -- so scale down by magnitude."""
    c = col.cast("double")
    return (F.when(c.isNull(), F.lit(None).cast("double"))
             .when(c >= 1e17, c / 1e9)   # nanoseconds  -> s
             .when(c >= 1e14, c / 1e6)   # microseconds -> s
             .when(c >= 1e11, c / 1e3)   # milliseconds -> s
             .otherwise(c))              # already seconds


def soft_deleted(df, deleted_at="deleted_at", deleted="deleted"):
    """Uniform soft-delete rule across products / variants / hotel rooms.

    A row counts as deleted when EITHER:
      * an explicit `deleted` flag is truthy (bool true or "1"/"true"), OR
      * `deleted_at` is set AND already in the PAST. A FUTURE deleted_at is a
        scheduled deletion -> the row is still live (NOT deleted) until then.
    Missing columns degrade gracefully (treated as absent). Never returns null
    so the derived is_deleted_* / is_bookable flags stay strictly boolean."""
    now = F.unix_timestamp(F.current_timestamp()).cast("double")
    da = optcol(df, deleted_at, "long")
    cond = da.isNotNull() & (_epoch_to_seconds(da) <= now)
    if deleted in df.columns:
        cond = cond | F.coalesce(F.col(deleted).cast("string"), F.lit("0")).isin(
            "1", "true", "True", "TRUE")
    return F.coalesce(cond, F.lit(False))


def json_text(df, col: str):
    """Return `col` as a JSON-text Column regardless of how it is stored.

    Bronze may land nested JSON as a STRING (already JSON text) or as a
    parsed struct/map. We must NOT to_json() a STRING (double-encodes), so
    branch on the stored type. Missing column -> NULL."""
    if col not in df.columns:
        return F.lit(None).cast("string")
    dt = df.schema[col].dataType
    return F.col(col) if isinstance(dt, StringType) else F.to_json(F.col(col))


def map_col(code_col, mapping: dict, default_col):
    """Chained CASE WHEN over a python dict, with a column default."""
    c = default_col
    for k, v in mapping.items():
        c = F.when(code_col == F.lit(k), F.lit(v)).otherwise(c)
    return c


# Mappings lifted verbatim from gold_to_graph.py so labels/axes match the KB.
_ATTR_EN_LABEL = {
    "holiday:swticket":        "ticket",
    "holiday:use_date":        "use_date",
    "holiday:hotel":           "hotel",
    "holiday:quantity":        "quantity",
    "holiday:age_group":       "age_group",
    "holiday:zone":            "zone",
    "holiday:depature":        "departure",
    "holiday:customer_object": "customer_type",
}
_VARIANT_AXIS_MAP = {
    "ticket":    "variant_tickets",
    "use_date":  "variant_dates",
    "hotel":     "variant_hotels",
    "quantity":  "variant_age_groups",
    "age_group": "variant_age_groups",
    "zone":      "variant_room_types",
    "departure": "variant_departures",
}
AXES_NO_STATUS = ["variant_tickets", "variant_dates", "variant_hotels",
                  "variant_age_groups", "variant_room_types", "variant_departures"]


def en_label_col(code_col):
    # default = substring after the last ':'  (e.g. "holiday:swticket" -> "swticket")
    default = F.element_at(F.split(code_col, ":"), -1)
    return map_col(code_col, _ATTR_EN_LABEL, default)


def axis_col(en_label_column):
    return map_col(en_label_column, _VARIANT_AXIS_MAP, F.lit(None).cast("string"))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 4, Finished, Available, Finished, False)

In [3]:
# --- Label lookups: attribute code -> vi/en names, value code -> vi name ---
attrs_raw   = read_src("product_attributes")
attr_tr_raw = read_src("product_attribute_translations")
vals_raw    = read_src("product_attribute_values")
val_tr_raw  = read_src("product_attribute_value_translations")

attrs = attrs_raw.select(F.col("id").alias("attr_id"), F.col("code").alias("attr_code"))
attr_tr = (attr_tr_raw.where(F.col("locale") == LOCALE)
           .select(F.col("attribute_id").alias("attr_id"), F.col("name").alias("attr_name_vi")))
# attr_code -> attr_name_vi (en label derived from the code itself)
attr_lbl = attrs.join(attr_tr, "attr_id", "left").select("attr_code", "attr_name_vi")

vals = vals_raw.select(F.col("id").alias("val_id"), F.col("code").alias("value_code"))
# NB: the value-translation join key lives in the (mis-named) `attribute_id`
# column and references product_attribute_values.id  (verified 30/30).
val_tr = (val_tr_raw.where(F.col("locale") == LOCALE)
          .select(F.col("attribute_id").alias("val_id"), F.col("name").alias("value_name_vi")))
val_lbl = vals.join(val_tr, "val_id", "left").select("value_code", "value_name_vi")

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 5, Finished, Available, Finished, False)

In [4]:
# --- Label lookups: attribute code -> vi/en names, value code -> vi name ---
attrs_raw   = read_src("product_attributes")
attr_tr_raw = read_src("product_attribute_translations")
vals_raw    = read_src("product_attribute_values")
val_tr_raw  = read_src("product_attribute_value_translations")

attrs = attrs_raw.select(F.col("id").alias("attr_id"), F.col("code").alias("attr_code"))
attr_tr = (attr_tr_raw.where(F.col("locale") == LOCALE)
           .select(F.col("attribute_id").alias("attr_id"), F.col("name").alias("attr_name_vi")))
# attr_code -> attr_name_vi (en label derived from the code itself)
attr_lbl = attrs.join(attr_tr, "attr_id", "left").select("attr_code", "attr_name_vi")

vals = vals_raw.select(F.col("id").alias("val_id"), F.col("code").alias("value_code"))
# NB: the value-translation join key lives in the (mis-named) `attribute_id`
# column and references product_attribute_values.id  (verified 30/30).
val_tr = (val_tr_raw.where(F.col("locale") == LOCALE)
          .select(F.col("attribute_id").alias("val_id"), F.col("name").alias("value_name_vi")))
val_lbl = vals.join(val_tr, "val_id", "left").select("value_code", "value_name_vi")

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 6, Finished, Available, Finished, False)

In [5]:
# --- Variants: base columns + soft-delete flag ---
v_raw = read_src("product_variants")

# Soft-delete rule (shared `soft_deleted` helper): an explicit `deleted` flag
# truthy OR a `deleted_at` already in the PAST (future = scheduled, still live).
del_col = soft_deleted(v_raw)

v_base = v_raw.select(
    F.col("id").alias("variant_id"),
    F.col("product_id"),
    optcol(v_raw, "status").alias("variant_status"),
    optcol(v_raw, "external_id").alias("variant_external_id"),
    json_text(v_raw, "attributes").alias("variant_attributes_json"),
    json_text(v_raw, "metadata").alias("variant_metadata_json"),
    optcol(v_raw, "created_at", "long").alias("variant_created_at"),
    optcol(v_raw, "updated_at", "long").alias("variant_updated_at"),
    optcol(v_raw, "deleted_at", "long").alias("variant_deleted_at"),
    del_col.alias("is_deleted_variant"),
)

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 7, Finished, Available, Finished, False)

In [6]:
# --- Explode variant attributes and resolve labels ---
vm = v_base.select(
    "variant_id", "product_id", "is_deleted_variant", "variant_status",
    F.from_json("variant_attributes_json", MapType(StringType(), StringType())).alias("attr_map"),
)
ve = (vm.select("variant_id", "product_id", "is_deleted_variant", "variant_status",
                F.explode_outer("attr_map").alias("attr_code", "value_code"))
        .join(attr_lbl, "attr_code", "left")
        .join(val_lbl, "value_code", "left")
        .withColumn("attr_name_en", en_label_col(F.col("attr_code")))
        .withColumn("axis", axis_col(F.col("attr_name_en"))))

# Per-variant: en attribute map + fully-labeled array (mirrors variants_struct.attributes)
attrs_per_variant = (ve.where(F.col("attr_code").isNotNull())
    .groupBy("variant_id")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(
                F.col("attr_name_en").alias("key"),
                F.col("value_name_vi").alias("value")))
        ).alias("attributes_en"),
        F.collect_list(F.struct(
            "attr_code", "attr_name_vi", "attr_name_en", "value_code", "value_name_vi"
        )).alias("attributes_labeled"),
    ))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 8, Finished, Available, Finished, False)

In [7]:
# --- Variants: base columns + soft-delete flag ---
v_raw = read_src("product_variants")

# Soft-delete rule (shared `soft_deleted` helper): an explicit `deleted` flag
# truthy OR a `deleted_at` already in the PAST (future = scheduled, still live).
del_col = soft_deleted(v_raw)

v_base = v_raw.select(
    F.col("id").alias("variant_id"),
    F.col("product_id"),
    optcol(v_raw, "status").alias("variant_status"),
    optcol(v_raw, "external_id").alias("variant_external_id"),
    json_text(v_raw, "attributes").alias("variant_attributes_json"),
    json_text(v_raw, "metadata").alias("variant_metadata_json"),
    optcol(v_raw, "created_at", "long").alias("variant_created_at"),
    optcol(v_raw, "updated_at", "long").alias("variant_updated_at"),
    optcol(v_raw, "deleted_at", "long").alias("variant_deleted_at"),
    del_col.alias("is_deleted_variant"),
)

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 9, Finished, Available, Finished, False)

In [8]:
# --- Explode variant attributes and resolve labels ---
vm = v_base.select(
    "variant_id", "product_id", "is_deleted_variant", "variant_status",
    F.from_json("variant_attributes_json", MapType(StringType(), StringType())).alias("attr_map"),
)
ve = (vm.select("variant_id", "product_id", "is_deleted_variant", "variant_status",
                F.explode_outer("attr_map").alias("attr_code", "value_code"))
        .join(attr_lbl, "attr_code", "left")
        .join(val_lbl, "value_code", "left")
        .withColumn("attr_name_en", en_label_col(F.col("attr_code")))
        .withColumn("axis", axis_col(F.col("attr_name_en"))))

# Per-variant: en attribute map + fully-labeled array (mirrors variants_struct.attributes)
attrs_per_variant = (ve.where(F.col("attr_code").isNotNull())
    .groupBy("variant_id")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(
                F.col("attr_name_en").alias("key"),
                F.col("value_name_vi").alias("value")))
        ).alias("attributes_en"),
        F.collect_list(F.struct(
            "attr_code", "attr_name_vi", "attr_name_en", "value_code", "value_name_vi"
        )).alias("attributes_labeled"),
    ))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 10, Finished, Available, Finished, False)

In [9]:
# --- Product-level rollups: variant axes + statuses + counts ---
# Axes are built over NON-soft-deleted variants (matches `mine` in load_products).
ax = (ve.where(~F.col("is_deleted_variant") & F.col("axis").isNotNull())
        .groupBy("product_id", "axis")
        .agg(F.collect_set("value_name_vi").alias("vals")))
ax_p = ax.groupBy("product_id").pivot("axis", AXES_NO_STATUS).agg(F.first("vals"))

ax_st = (v_base.where(~F.col("is_deleted_variant"))
         .groupBy("product_id")
         .agg(F.collect_set("variant_status").alias("variant_statuses")))

counts = (v_base.where(~F.col("is_deleted_variant"))
          .groupBy("product_id")
          .agg(F.count("*").alias("variant_count"),
               F.sum(F.when(F.col("variant_status") == "active", 1).otherwise(0))
                .alias("active_variant_count")))

# use_date validity: parse the per-variant use_date attribute, roll up to the
# product. max_use_date drives the is_bookable expiry gate in the assemble cell.
# NOTE: value format assumed ISO yyyy-MM-dd. If value_name_vi is dd/MM/yyyy use
#       F.to_date(F.col("value_name_vi"), "dd/MM/yyyy") instead.
use_dates = (ve.where((F.col("attr_name_en") == "use_date") & ~F.col("is_deleted_variant"))
               .withColumn("use_date_parsed", F.to_date(F.col("value_name_vi")))
               .groupBy("product_id")
               .agg(F.max("use_date_parsed").alias("max_use_date"),
                    F.count("*").alias("use_date_count")))

product_rollup = (ax_p.join(ax_st, "product_id", "outer")
                      .join(counts, "product_id", "outer")
                      .join(use_dates, "product_id", "outer"))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 11, Finished, Available, Finished, False)

In [10]:
# --- POI bridge: product -> exactly one business_service_id ---------------
# A product with more than one distinct non-null relation is ambiguous: the
# contract permits only one source_id, so it is excluded rather than arbitrarily
# choosing a POI. Products with exactly one relation use that ID; products with no
# relation can still use the product-post business_service_id fallback below.
_rel_raw = read_src("product_business_service_relation").where(F.col("business_service_id").isNotNull())
_rel_counts = _rel_raw.groupBy("product_id").agg(F.countDistinct("business_service_id").alias("_poi_link_count"))
ambiguous_product_ids = _rel_counts.where(F.col("_poi_link_count") > 1).select("product_id")
rel = (_rel_raw.join(_rel_counts, "product_id", "inner")
       .where(F.col("_poi_link_count") == 1)
       .groupBy("product_id")
       .agg(F.first("business_service_id", ignorenulls=True).alias("business_service_id")))

# --- Variant translations (vi name) ---
vt_raw = read_src("product_variant_translations")
vt = (vt_raw.where(F.col("locale") == LOCALE)
      .select(F.col("variant_id"),
              optcol(vt_raw, "name").alias("variant_name_vi"),
              optcol(vt_raw, "description").alias("variant_description_vi")))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 12, Finished, Available, Finished, False)

In [11]:
# --- Products + metadata extraction + vi name/description ---
p_raw = read_src("products")
p = (p_raw.select(
        F.col("id").alias("product_id"),
        optcol(p_raw, "product_line").alias("product_line"),
        optcol(p_raw, "status").alias("product_status"),
        optcol(p_raw, "external_id").alias("product_external_id"),
        json_text(p_raw, "attributes").alias("product_attributes_json"),
        json_text(p_raw, "metadata").alias("product_metadata_json"),
        optcol(p_raw, "created_at", "long").alias("product_created_at"),
        optcol(p_raw, "updated_at", "long").alias("product_updated_at"),
        optcol(p_raw, "deleted_at", "long").alias("product_deleted_at"),
        soft_deleted(p_raw).alias("is_deleted_product"),
    )
    .withColumn("holiday_product_post_id",
                F.coalesce(F.get_json_object("product_metadata_json", "$.holiday_product_post_id"),
                           F.get_json_object("product_metadata_json", "$.product_post_id")))
    .withColumn("main_attribute_code", F.get_json_object("product_metadata_json", "$.main_attribute_code"))
    .withColumn("product_start_date",  F.get_json_object("product_metadata_json", "$.start_date"))
    .withColumn("product_end_date",    F.get_json_object("product_metadata_json", "$.end_date")))

pt_raw = read_src("product_translations")
pt = (pt_raw.where(F.col("locale") == LOCALE)
      .select(F.col("product_id"),
              optcol(pt_raw, "name").alias("product_name_vi"),
              optcol(pt_raw, "description").alias("product_description_vi")))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 13, Finished, Available, Finished, False)

In [12]:
# --- Posts (pricing + content) ---
# shortDescription + tags come from the post's raw_sync_data JSON (the
# product_post_translations table is empty in Fabric). The English
# shortDescription is pre-provided under translations[languageCode="en-VN"],
# so no Azure translation is needed for it (fall back to vi, then top-level).
po_raw = read_src("product_posts")

import json as _pj
def _post_short_desc(rsd):
    try: d = _pj.loads(rsd) if rsd else None
    except Exception: return None
    if not isinstance(d, dict): return None
    en = vi = None
    for t in (d.get("translations") or []):
        if not isinstance(t, dict): continue
        lc = (t.get("languageCode") or "").lower()
        sd = t.get("shortDescription")
        if lc.startswith("en"): en = en or sd
        elif lc.startswith("vi"): vi = vi or sd
    return en or vi or d.get("shortDescription")
def _post_tags(rsd):
    try: d = _pj.loads(rsd) if rsd else None
    except Exception: return None
    if not isinstance(d, dict): return None
    labels = [(t.get("label") or t.get("code")) for t in (d.get("tags") or [])
              if isinstance(t, dict) and (t.get("label") or t.get("code"))]
    return _pj.dumps(labels, ensure_ascii=False) if labels else None
_short_desc_udf = F.udf(_post_short_desc, StringType())
_tags_udf = F.udf(_post_tags, StringType())
def _post_name(rsd):
    try: d = _pj.loads(rsd) if rsd else None
    except Exception: return None
    if not isinstance(d, dict): return None
    vi = None
    for t in (d.get("translations") or []):
        if isinstance(t, dict) and (t.get("languageCode") or "").lower().startswith("vi"):
            vi = vi or t.get("name")
    return vi or d.get("name")
_post_name_udf = F.udf(_post_name, StringType())

# NB: product -> post links on post.external_id (= products.metadata.holiday_product_post_id).
po = (po_raw.select(
        F.col("id").alias("product_post_internal_id"),
        optcol(po_raw, "external_id").alias("product_post_external_id"),
        optcol(po_raw, "status").alias("post_status"),
        optcol(po_raw, "business_service_id").alias("post_business_service_id"),
        decode_price(po_raw, "original_price").alias("original_price"),
        decode_price(po_raw, "marketing_price").alias("marketing_price"),
        decode_markup(po_raw, "markup_percent").alias("markup_percent"),
        optcol(po_raw, "thumbnail_url").alias("thumbnail_url"),
        json_text(po_raw, "gallery").alias("gallery_json"),
        optcol(po_raw, "validity_from").alias("validity_from"),
        optcol(po_raw, "validity_to").alias("validity_to"),
        _short_desc_udf(json_text(po_raw, "raw_sync_data")).alias("post_short_description_en"),
        _tags_udf(json_text(po_raw, "raw_sync_data")).alias("post_tags_json"),
        _post_name_udf(json_text(po_raw, "raw_sync_data")).alias("post_title_vi"),
        optcol(po_raw, "deleted_at", "long").alias("post_deleted_at"),
    )
    .withColumn("is_deleted_post", F.col("post_deleted_at").isNotNull())
    .withColumn("_p", F.coalesce("marketing_price", "original_price"))
    .withColumn("price_level",
                F.when(F.col("_p").isNull(), F.lit(None).cast("int"))
                 .when(F.col("_p") <= 500000, 1)
                 .when(F.col("_p") <= 2000000, 2)
                 .when(F.col("_p") <= 5000000, 3)
                 .otherwise(4))
    .drop("_p"))

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 14, Finished, Available, Finished, False)

In [13]:
# --- Assemble one enriched row per variant ---
final = (v_base
    # Exclude ambiguous product→POI links before any post fallback can apply.
    .join(ambiguous_product_ids, "product_id", "left_anti")
    .join(vt, "variant_id", "left")
    .join(attrs_per_variant, "variant_id", "left")
    .join(p, "product_id", "left")
    .join(pt, "product_id", "left")
    .join(rel, "product_id", "left")
    .join(product_rollup, "product_id", "left")
    .join(po, p.holiday_product_post_id == po.product_post_external_id, "left"))

# --- Searchability gates -----------------------------------------------------
# R1: DIFF products must be linked to an ACTIVE product_post (no post / inactive
#     post -> not searchable). Non-DIFF products are unaffected.
# R2: if a product carries a use_date, its latest use_date must be today or
#     later; otherwise it is expired and not searchable. Products with no
#     use_date are unaffected.
# Use-date validity belongs to is_bookable, not is_active: an expired product
# can remain visible in the catalog but cannot be selected for booking.
_post_active = (F.coalesce(F.col("post_status") == "active", F.lit(False))
                & ~F.coalesce(F.col("is_deleted_post"), F.lit(False)))
_has_use_date = F.coalesce(F.col("use_date_count"), F.lit(0)) > 0
_use_date_ok  = F.coalesce(F.col("max_use_date") >= F.current_date(), F.lit(False))
# With HIDE_EXPIRED_USE_DATE=False, this gate is always true.
_use_date_gate = (~_has_use_date | _use_date_ok) if HIDE_EXPIRED_USE_DATE else F.lit(True)

final = (final
    # product_post_id == the value products carry (post.external_id)
    .withColumn("product_post_id", F.col("holiday_product_post_id"))
    .withColumn("product_effective_status",
                F.when(F.col("is_deleted_product"), F.lit("deleted")).otherwise(F.col("product_status")))
    # Catalog lifecycle only: status and soft deletion, independent of use date.
    .withColumn("is_active",
                (F.col("product_status") == "active") & ~F.col("is_deleted_product"))
    # single scalar bridge id: relation id, else the post's business_service_id
    .withColumn("business_service_id",
                F.coalesce(F.col("business_service_id"), F.col("post_business_service_id"))))

# --- Holiday bookability ----------------------------------------------------
# DIFF is detected from the accent-folded Vietnamese product name/post title only
# to apply its active-post requirement. POI linkage is resolved later from the
# relation-table business_service_id (or the post fallback), not from name matching.
_match_text = fold_udf(F.concat_ws(" ",
    F.coalesce(F.col("product_name_vi"), F.lit("")),
    F.coalesce(F.col("post_title_vi"), F.lit(""))))
def _contains_any(text_col, aliases):
    condition = F.lit(False)
    for alias in aliases:
        alias = _fold(alias).strip()
        if alias:
            condition = condition | (F.instr(text_col, alias) > 0)
    return condition
_is_diff_product = _contains_any(_match_text, ["DIFF","diff"])
_has_active_variant = F.coalesce(F.col("active_variant_count"), F.lit(0)) > 0
_diff_post_ok = ((~_is_diff_product) | _post_active) if REQUIRE_ACTIVE_POST_FOR_DIFF else F.lit(True)
final = (final
    .withColumn("source_id", F.lit(None).cast("string"))
    # is_active is lifecycle-only; use date, variant, and DIFF-post checks belong here.
    .withColumn("is_bookable", F.col("is_active") & _use_date_gate
                & _has_active_variant & _diff_post_ok))
# Column ordering + the final select() happen in the Write cell, AFTER the
# translation cell has added the English columns.

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 15, Finished, Available, Finished, False)

In [14]:
# --- vi -> en translation (Azure OpenAI / GPT-4.1) with a PERSISTED cache ------
# product_name is translated to English; the post shortDescription is ALREADY
# English in raw_sync_data (translations[en-VN]) — set in the posts cell — so it
# is NOT translated here. Variant names + attribute values stay Vietnamese.
#
# Cache: translations live in TRANSLATION_TABLE, keyed by product_id. Each run we
# translate ONLY product_ids missing from that table, append the new ones back,
# then attach product_name_en from the table. A product_id already translated is
# never re-sent to GPT. The SAME function is reused by the hotel cell, so holiday
# + hotel room names share one cache.
#
# Throughput: names are translated in BATCHES (many per request) with a pause
# between batches + retry/backoff, so we don't trip Azure OpenAI rate limits.
# A non-retryable error (e.g. HTTP 400 content-filter on one name) falls back to
# the vi text for that item and is cached, so it isn't retried next run.
#
# NOTE: keyed by product_id only — if a product's vi name later changes it is NOT
# re-translated. To force a refresh, delete its row from TRANSLATION_TABLE.
import json as _json
import time
import urllib.request
import urllib.error

_cache_schema = "product_id string, product_name_vi string, product_name_en string"


def _chat(messages, max_retries=None):
    """One Azure OpenAI chat call with retry + exponential backoff. Returns text or None.
    Prints the response body on a non-retryable HTTP error so the cause is visible."""
    max_retries = TRANSLATE_MAX_RETRIES if max_retries is None else max_retries
    url = (f"{AZURE_AI_ENDPOINT}/openai/deployments/{AZURE_AI_DEPLOYMENT}"
           f"/chat/completions?api-version={AZURE_AI_API_VERSION}")
    data = _json.dumps({"temperature": 0, "messages": messages}).encode("utf-8")
    base = 2.0
    for attempt in range(max_retries):
        req = urllib.request.Request(
            url, data=data,
            headers={"api-key": AZURE_AI_API_KEY, "Content-Type": "application/json"})
        try:
            resp = _json.loads(urllib.request.urlopen(req, timeout=120).read())
            return (resp["choices"][0]["message"]["content"] or "").strip()
        except urllib.error.HTTPError as e:
            body = ""
            try: body = e.read().decode("utf-8", "ignore")[:300]
            except Exception: pass
            # 429 / 5xx are transient -> back off and retry; 400/401/403 are not.
            if e.code in (429, 500, 502, 503, 504) and attempt < max_retries - 1:
                wait = base * (2 ** attempt)
                print(f"  HTTP {e.code} — retry {attempt + 1}/{max_retries} in {wait:.0f}s")
                time.sleep(wait)
                continue
            print(f"  translate HTTP {e.code}:", body or "(no body)")
            return None
        except Exception as e:
            if attempt < max_retries - 1:
                wait = base * (2 ** attempt)
                print(f"  error {str(e)[:80]} — retry {attempt + 1}/{max_retries} in {wait:.0f}s")
                time.sleep(wait)
                continue
            print("  translate failed:", str(e)[:140])
            return None
    return None


_TR_MEMO = {}
def translate_one(text):
    """Translate a single Vietnamese string (fallback path). Memoized in-process."""
    if text is None or not str(text).strip():
        return text
    if text in _TR_MEMO:
        return _TR_MEMO[text]
    out = _chat([
        {"role": "system", "content":
            "You are a translator. Translate the user's Vietnamese text to natural "
            "English. Return ONLY the translation - no quotes, no notes. If the text "
            "is already English, return it unchanged."},
        {"role": "user", "content": str(text)}]) or text
    _TR_MEMO[text] = out
    return out

translate_vi_en = translate_one  # backward-compatible alias


def translate_batch(texts):
    """Translate a list of Vietnamese strings in ONE request.
    Asks GPT for a JSON array [{"id","translation"}] so items stay aligned;
    falls back to per-item (delayed) translation if the response can't be parsed."""
    items = [{"id": i, "text": str(t)} for i, t in enumerate(texts)]
    out = _chat([
        {"role": "system", "content":
            "You are a translator. The user sends a JSON array of objects with 'id' "
            "and Vietnamese 'text'. Translate each 'text' to natural English. Return "
            "ONLY a JSON array of objects [{\"id\": <int>, \"translation\": <string>}] "
            "with the SAME ids and nothing else. If a text is already English, return "
            "it unchanged."},
        {"role": "user", "content": _json.dumps(items, ensure_ascii=False)}])
    if out:
        try:
            s = out[out.index("["): out.rindex("]") + 1]
            parsed = {int(o["id"]): o["translation"] for o in _json.loads(s)}
            res = [parsed.get(i) for i in range(len(texts))]
            if all(r is not None for r in res):
                return res
        except Exception as e:
            print("  batch parse failed — falling back per-item:", str(e)[:100])
    res = []
    for t in texts:
        res.append(translate_one(t))
        time.sleep(TRANSLATE_DELAY_SEC)
    return res


def translate_product_names(df):
    """Attach product_name_en to df (needs product_id + product_name_vi) using the
    persisted TRANSLATION_TABLE cache. Translates only product_ids not already
    cached (batched + delayed), appends the new ones, then joins en back. Reused
    by BOTH the holiday (`final`) and hotel (`hotel_prod`) builds."""
    need = (df.select("product_id", "product_name_vi")
              .where(F.col("product_name_vi").isNotNull())
              .dropDuplicates(["product_id"]))
    try:
        cache_df = spark.read.table(TRANSLATION_TABLE)
        print(f"translation cache: {cache_df.count()} rows in {TRANSLATION_TABLE}")
    except Exception:
        cache_df = spark.createDataFrame([], _cache_schema)
        print(f"translation cache: {TRANSLATION_TABLE} not found yet — will create it")

    todo = (need.join(cache_df.select("product_id"), "product_id", "left_anti")
                .collect())
    print(f"translation: {len(todo)} new product_ids to translate "
          f"(batch={TRANSLATE_BATCH_SIZE}, delay={TRANSLATE_DELAY_SEC}s, "
          f"batch_pause={TRANSLATE_BATCH_PAUSE_SEC}s)")

    new_rows = []
    if todo:
        if ENABLE_TRANSLATION and AZURE_AI_API_KEY:
            for b in range(0, len(todo), TRANSLATE_BATCH_SIZE):
                chunk = todo[b: b + TRANSLATE_BATCH_SIZE]
                ens = translate_batch([r["product_name_vi"] for r in chunk])
                for r, en in zip(chunk, ens):
                    new_rows.append((r["product_id"], r["product_name_vi"],
                                     en or r["product_name_vi"]))
                print(f"  translated {min(b + TRANSLATE_BATCH_SIZE, len(todo))}/{len(todo)}")
                time.sleep(TRANSLATE_BATCH_PAUSE_SEC)
        else:
            print("translation disabled — caching vi text as the en fallback.")
            for r in todo:
                new_rows.append((r["product_id"], r["product_name_vi"], r["product_name_vi"]))

    if new_rows:
        new_df = (spark.createDataFrame(new_rows, _cache_schema)
                       .withColumn("updated_at", F.current_timestamp()))
        (new_df.write.format("delta").mode("append")
               .option("mergeSchema", "true").saveAsTable(TRANSLATION_TABLE))
        print(f"appended {len(new_rows)} translations -> {TRANSLATION_TABLE}")

    try:
        full_tr = (spark.read.table(TRANSLATION_TABLE)
                        .select("product_id", F.col("product_name_en").alias("_tr_en"))
                        .dropDuplicates(["product_id"]))
        return (df.join(full_tr, "product_id", "left")
                  .withColumn("product_name_en",
                              F.coalesce(F.col("_tr_en"), F.col("product_name_vi")))
                  .drop("_tr_en"))
    except Exception as e:
        print("translation read-back skipped:", str(e)[:120], "— en falls back to vi.")
        return df.withColumn("product_name_en", F.col("product_name_vi"))


# Holiday products: translate names via the cache.
final = translate_product_names(final)
print("holiday translation done.")


StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 16, Finished, Available, Finished, False)

translation cache: 236 rows in lh_vv_gold.dbo.ai_product_translation
translation: 0 new product_ids to translate (batch=20, delay=0.5s, batch_pause=3.0s)
holiday translation done.


In [15]:
# --- Hotel inventory -> products (room type = product, rate plans = variants) ---
# Two bronze partner tables keyed by business_service_id (the hotel = a gold POI):
#   hotel_room_types  -> one product per room category (the sellable thing:
#                        name, occupancy, size, view, amenities, availability)
#   hotel_rate_plans  -> the hotel's pricing/policy plans; nested as the room's
#                        variants. Rate plans attach at the PROPERTY level (no
#                        room<->rate key in the source), so every room of a hotel
#                        carries that hotel's full rate-plan list. (Unlike holiday
#                        ticket variants, which are scoped per-product, these are
#                        shared across all of a hotel's rooms -- an accepted
#                        approximation until a room<->rate link table exists.)
# Output is one row per room at the SAME product grain as the holiday `prod`
# (built in the write cell), so it unions straight in and inherits the POI
# resolution / null-POI drop / enriched-POI validation. No price in these tables
# -> original_price / marketing_price / markup_percent / price_level stay NULL.
# Navigation parallel: POI(hotel) -> product(room) -> variant(rate plan).
# The room IS the product, so room type is NOT a variant axis: variant_room_types
# is left NULL for hotels; the room's identity lives in `name` + product_attributes.
hotel_prod = None
if ENABLE_HOTEL:
    rt_raw = read_src("hotel_room_types")
    rp_raw = read_src("hotel_rate_plans")

    # -- Rate plans -> per-hotel variant list -------------------------------
    _MEAL = {"0": "Room only", "1": "Breakfast", "2": "Half board",
             "3": "Full board", "4": "All inclusive"}
    rp = (rp_raw.select(
            F.col("id").alias("variant_id"),
            F.col("business_service_id").alias("business_service_id"),
            optcol(rp_raw, "name").alias("rp_name"),
            optcol(rp_raw, "status").alias("rp_status"),
            optcol(rp_raw, "kind").alias("rp_kind"),
            optcol(rp_raw, "meal_plan").alias("rp_meal_plan"),
            optcol(rp_raw, "valid_from", "long").cast("string").alias("rp_valid_from"),
            optcol(rp_raw, "valid_to", "long").cast("string").alias("rp_valid_to"),
            optcol(rp_raw, "include_tax").cast("string").alias("rp_include_tax"),
            optcol(rp_raw, "include_service_charge").cast("string").alias("rp_include_service_charge"),
            soft_deleted(rp_raw).alias("rp_deleted"),
        )
        .withColumn("rp_meal_label", map_col(F.col("rp_meal_plan"), _MEAL, F.col("rp_meal_plan"))))

    # variant struct mirrors the holiday variant shape: {variant_id, name, status, attributes(map)}
    _rate_variant = F.struct(
        F.col("variant_id").alias("variant_id"),
        F.coalesce(F.col("rp_name"), F.col("variant_id")).alias("name"),
        F.col("rp_status").alias("status"),
        F.map_from_arrays(
            F.array(*[F.lit(k) for k in ["meal_plan", "kind", "valid_from", "valid_to",
                                         "include_tax", "include_service_charge"]]),
            F.array(F.col("rp_meal_label"), F.col("rp_kind"), F.col("rp_valid_from"),
                    F.col("rp_valid_to"), F.col("rp_include_tax"),
                    F.col("rp_include_service_charge")),
        ).alias("attributes"),
    )
    rates_by_hotel = (rp.where(~F.col("rp_deleted"))
        .groupBy("business_service_id")
        .agg(
            F.to_json(F.collect_list(_rate_variant)).alias("rate_variants_json"),
            F.count("*").alias("rate_count"),
            F.sum(F.when(F.col("rp_status") == F.lit("Active"), 1).otherwise(0)).alias("active_rate_count"),
        ))

    # -- Room types -> products ---------------------------------------------
    rt = (rt_raw.select(
            F.col("id").alias("product_id"),
            F.col("business_service_id").alias("business_service_id"),
            optcol(rt_raw, "name").alias("product_name_vi"),
            optcol(rt_raw, "description").alias("room_description"),
            optcol(rt_raw, "status").alias("room_status"),
            optcol(rt_raw, "max_occupancy").cast("int").alias("room_occupancy"),
            optcol(rt_raw, "adult_capacity").cast("int").alias("room_adult_capacity"),
            optcol(rt_raw, "child_capacity").cast("int").alias("room_child_capacity"),
            optcol(rt_raw, "max_extra_bed").cast("int").alias("room_max_extra_bed"),
            optcol(rt_raw, "room_size").cast("int").alias("room_size"),
            optcol(rt_raw, "view_type").alias("room_view_type"),
            optcol(rt_raw, "available_rooms").cast("int").alias("room_available"),
            optcol(rt_raw, "total_rooms").cast("int").alias("room_total"),
            json_text(rt_raw, "amenities").alias("amenities_json"),
            json_text(rt_raw, "tags").alias("tags_json"),
            soft_deleted(rt_raw).alias("is_deleted_product"),
        )
        # bronze uses the literal string "NULL" (and blanks) as a no-view sentinel
        .withColumn("room_view_type",
                    F.when(F.upper(F.coalesce(F.col("room_view_type"), F.lit(""))).isin("NULL", ""),
                           F.lit(None).cast("string")).otherwise(F.col("room_view_type")))
        .withColumn("amenity_names",
                    F.expr("transform(from_json(amenities_json, 'array<struct<amenityName:string>>'), "
                           "x -> x.amenityName)"))
        .withColumn("is_active", (F.col("room_status") == F.lit("Active")) & ~F.col("is_deleted_product"))
        .withColumn("product_effective_status",
                    F.when(F.col("is_deleted_product"), F.lit("deleted")).otherwise(F.col("room_status")))
        # bookable: active room, linked to a hotel, with rooms left to sell
        .withColumn("is_bookable",
                    F.col("is_active") & F.col("business_service_id").isNotNull()
                    & (F.coalesce(F.col("room_available"), F.lit(0)) > 0)))

    hotel = rt.join(rates_by_hotel, "business_service_id", "left")

    # product_attributes JSON, prebuilt for hotels (the write cell uses it as-is).
    # shortDescription feeds the chunk vector; tags -> payload subcategory_tags;
    # the room specifics ride alongside so a room product is self-describing.
    # viewType here is the room's own attribute -> room type identity lives on the
    # PRODUCT, not in the variant_room_types axis (which stays NULL for hotels).
    _hotel_attrs = F.to_json(F.struct(
        F.col("room_description").alias("shortDescription"),
        F.from_json(F.col("tags_json"), ArrayType(StringType())).alias("tags"),
        F.lit(None).cast("int").alias("priceLevel"),          # no price source
        F.col("is_active").alias("isActive"),
        F.col("room_occupancy").alias("maxOccupancy"),
        F.col("room_adult_capacity").alias("adultCapacity"),
        F.col("room_child_capacity").alias("childCapacity"),
        F.col("room_max_extra_bed").alias("maxExtraBed"),
        F.col("room_size").alias("roomSize"),
        F.col("room_view_type").alias("viewType"),
        F.col("amenity_names").alias("amenities"),
        F.col("room_available").alias("availableRooms"),
        F.col("room_total").alias("totalRooms"),
    ))

    hotel_prod = (hotel
        .withColumn("variants_json", F.coalesce(F.col("rate_variants_json"), F.lit("[]")))
        .withColumn("product_line", F.lit("accommodation"))
        .withColumn("product_type", F.lit("hotel"))
        .withColumn("variant_count", F.coalesce(F.col("rate_count"), F.lit(0)).cast("long"))
        .withColumn("active_variant_count", F.coalesce(F.col("active_rate_count"), F.lit(0)).cast("long"))
        # NB: no variant axes for hotels -- room type is the product, not an axis.
        # variant_room_types (and the other axes) union in as NULL for hotel rows.
        .withColumn("price_level", F.lit(None).cast("int"))
        .withColumn("original_price", F.lit(None).cast("double"))
        .withColumn("marketing_price", F.lit(None).cast("double"))
        .withColumn("markup_percent", F.lit(None).cast("double"))
        .withColumn("post_short_description_en", F.col("room_description"))
        .withColumn("post_tags_json", F.col("tags_json"))
        .withColumn("source_id", F.lit(None).cast("string"))   # resolved by the POI block in the write cell
        .withColumn("updated_at", F.current_timestamp())
        .withColumn("prebuilt_product_attributes", _hotel_attrs)
        .select(
            "product_id", "variants_json", "product_name_vi", "product_line", "product_type",
            "product_effective_status", "is_active", "is_bookable",
            "variant_count", "active_variant_count",
            "original_price", "marketing_price", "markup_percent", "price_level",
            "post_short_description_en", "post_tags_json",
            "business_service_id", "source_id", "updated_at", "prebuilt_product_attributes",
        ))

    # vi -> en room names via the SAME persisted cache as the holiday names:
    # only un-cached product_ids are translated (batched + delayed) and appended
    # to TRANSLATION_TABLE. Room descriptions are left as-is (already English).
    hotel_prod = translate_product_names(hotel_prod)

    print(f"hotel products built: {hotel_prod.count()} rooms across "
          f"{hotel_prod.select('business_service_id').distinct().count()} hotels")

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 17, Finished, Available, Finished, False)

In [16]:
# --- Write the enriched gold table — EXACTLY the KB Product contract ---
# Output schema = product_field_mapping.md §A (25 columns). No duplicate/reference
# columns: the KB needs only these. Venue fields (business_sector/address/website_url/
# amenities) are hydrated from the linked POI at query time, so they are NOT here.

# Watermark for the Event Hub sync; ai_product is a full overwrite, so per-row
# Preserve the source product update time as a timestamp.
final = final.withColumn(
    "updated_at",
    _epoch_to_seconds(F.col("product_updated_at")).cast("timestamp")
)

# --- Collapse to PRODUCT grain (one row per product_id) ---
# Variants nested in variants_json; product-level columns are first() over the variants.
_variant_struct = F.struct(
    F.col("variant_id").alias("variant_id"),
    F.col("variant_name_vi").alias("name"),
    F.col("variant_status").alias("status"),
    F.coalesce(F.col("attributes_en"),
               F.create_map().cast(MapType(StringType(), StringType()))).alias("attributes"),
)
# Only the intermediates needed to derive the 25 contract columns below.
_PRODUCT_COLS = [
    "product_name_en", "product_name_vi", "product_line",
    "product_effective_status", "is_active", "is_bookable",
    "variant_count", "active_variant_count",
    "variant_tickets", "variant_dates", "variant_hotels", "variant_age_groups",
    "variant_room_types", "variant_departures", "variant_statuses",
    "original_price", "marketing_price", "markup_percent", "price_level",
    "post_short_description_en", "post_tags_json",
    "business_service_id", "source_id", "updated_at",
]
prod = final.groupBy("product_id").agg(
    F.to_json(F.collect_list(_variant_struct)).alias("variants_json"),
    *[F.first(F.col(c), ignorenulls=True).alias(c) for c in _PRODUCT_COLS],
)
# Holiday products: generic product_type, no prebuilt product_attributes.
prod = (prod
    .withColumn("product_type", F.lit("other"))
    .withColumn("prebuilt_product_attributes", F.lit(None).cast("string")))

# Fold hotel-room products (room=product, rate plans=variants) in at the same
# grain. unionByName aligns columns; holiday-only axes/price cols become NULL on
# hotel rows (and the hotel-only prebuilt attrs NULL on holiday rows). Hotels then
# ride the POI resolution / null-POI drop / enriched-POI validation below.
if ENABLE_HOTEL and hotel_prod is not None:
    prod = prod.unionByName(hotel_prod, allowMissingColumns=True)
    print(f"union: holiday + hotel = {prod.count()} products "
          f"({prod.where(F.col('product_type') == 'hotel').count()} hotel)")

# --- Resolve POI linkage: product.business_service_id IS the gold POI source_id ---
# product_business_service_relation maps product_id -> business_service_id, and that
# business_service_id is the POI's source_id (Google Place ID, the same key ai_poi /
# ai_enriched_poi / the POI graph node all use). So set source_id STRAIGHT from the
# bridge (the assemble step initializes source_id as null), instead of guessing the
# matching ai_poi column by value-overlap (that only rescued products whose id already
# happened to be present in ai_poi). ai_poi is then used ONLY to inherit destination_id
# for the products whose POI row exists. Cast keys to STRING so an int/string type 
# mismatch can't silently drop the join to null.
prod = prod.withColumn(
    "source_id",
    F.coalesce(F.col("source_id"), F.col("business_service_id").cast("string")))
try:
    _poi = spark.read.table(POI_GOLD_TABLE)
    _poi_link = (_poi.select(F.col("source_id").cast("string").alias("_poi_key"),
                             F.col("destination_id").alias("_poi_dest"))
                     .where(F.col("_poi_key").isNotNull())
                     .dropDuplicates(["_poi_key"]))
    _n_src = prod.where(F.col("source_id").isNotNull()).select("source_id").distinct().count()
    _matched = (prod.where(F.col("source_id").isNotNull())
                    .join(F.broadcast(_poi_link),
                          prod["source_id"].cast("string") == F.col("_poi_key"), "left_semi")
                    .count())
    print(f"POI linkage: source_id <- business_service_id; "
          f"{_matched}/{prod.where(F.col('source_id').isNotNull()).count()} products "
          f"match an ai_poi row for destination_id ({_n_src} distinct source_id)")
    prod = (prod.join(_poi_link, prod["source_id"].cast("string") == F.col("_poi_key"), "left")
                .withColumn("destination_id", F.col("_poi_dest"))
                .drop("_poi_key", "_poi_dest"))
                
except Exception as _e:
    print("POI linkage join skipped:", str(_e)[:200])
    if "destination_id" not in prod.columns:
        prod = prod.withColumn("destination_id", F.lit(None).cast("string"))

# --- Destination ID remap: source IDs -> KB Memgraph IDs ---
_destination_map = F.create_map(
    *[item for source_id, kb_id in DESTINATION_ID_REMAP.items()
      for item in (F.lit(source_id), F.lit(kb_id))]
)

prod = prod.withColumn(
    "destination_id",
    F.coalesce(
        F.element_at(_destination_map, F.col("destination_id").cast("string")),
        F.col("destination_id").cast("string"),
    ),
)

# --- Partner scope -----------------------------------------------------------
# No partner-specific filter is applied in this notebook. POI eligibility is
# enforced below by requiring source_id to exist in ai_enriched_poi.

# --- Derive the contract columns ---
final = (prod
    # Price defaults: pricing lives on product_posts, so products without a priced
    # post (DIFF-night / combo posts where price is per-variant, products with no
    # linked post, hotel rooms) carry NULL prices. Default the numeric price fields
    # AND price_level to 0 so the contract never ships null prices. Done first so the
    # 0 price_level also flows into the product_attributes JSON below.
    .withColumn("original_price", F.coalesce(F.col("original_price"), F.lit(0)).cast("double"))
    .withColumn("marketing_price", F.coalesce(F.col("marketing_price"), F.lit(0)).cast("double"))
    .withColumn("markup_percent", F.coalesce(F.col("markup_percent"), F.lit(0)).cast("double"))
    .withColumn("price_level", F.coalesce(F.col("price_level"), F.lit(0)).cast("int"))
    # smoke-test pin: in TEST_MODE force every product onto one POI that the push
    # also ships, so the AT_POI link + parent_poi hydration resolve. Off in prod;
    # otherwise source_id keeps the value inherited from the POI table above.
    .withColumn("source_id",
                F.lit(FORCE_PRODUCT_SOURCE_ID) if (TEST_MODE and FORCE_PRODUCT_SOURCE_ID) else F.col("source_id"))
    .withColumn("name", F.coalesce("product_name_en", "product_name_vi", "product_id"))
    # content shipped in this table is English (product_name_en + post_short_description_en).
    .withColumn("lang", F.lit("en"))
    # product_type already carried on prod ("other" for holiday, "hotel" for rooms).
    .withColumn("product_type", F.coalesce(F.col("product_type"), F.lit("other")))
    # source_id / destination_id are resolved above by business_service_id against
    # the gold POI table. SCALAR on purpose — the push casts columns to STRING and
    # id_list() cannot parse JSON-array strings.
    # product_attributes: shortDescription is embedded into the chunk vector; tags ->
    # payload subcategory_tags; priceLevel/isActive mirror the scalar fields. Hotel
    # rows carry a prebuilt product_attributes (room specifics); holiday rows fall
    # back to the generic struct.
    .withColumn("product_attributes", F.coalesce(
        F.col("prebuilt_product_attributes"),
        F.to_json(F.struct(
            F.col("post_short_description_en").alias("shortDescription"),
            F.from_json(F.col("post_tags_json"), ArrayType(StringType())).alias("tags"),
            F.col("price_level").alias("priceLevel"),
            F.col("is_active").alias("isActive"))))))

# Keep business_service_id around (it is NOT a contract column) only so the
# diagnostics below can show which hotel/product the dropped POI belongs to.
_diag = final.select("product_id", "name", "product_type", "product_line",
                     "business_service_id", "source_id", "destination_id")

# Axes -> JSON strings on the wire; the KB coerces them back to list[str].
for _axis in ("variant_tickets", "variant_dates", "variant_hotels", "variant_age_groups",
              "variant_room_types", "variant_departures", "variant_statuses"):
    final = final.withColumn(_axis, F.to_json(F.col(_axis)))

# Test mode: pin destination_id to the FORCE POI's destination (matches the
# pinned source_id), so the smoke-test product carries a real destination even
# if the gold-table join misses it.
if TEST_MODE and FORCE_PRODUCT_DESTINATION_ID:
    final = final.withColumn("destination_id", F.lit(FORCE_PRODUCT_DESTINATION_ID))

# TEMPORARY: a product that HAS a business_service_id but whose POI yielded no
# destination_id (POI row missing a destination, or no POI matched) falls back to
# Da Nang so it stays destination-filterable in the KB. Remove once every linked
# POI carries a real destination_id. Products with no business_service_id (e.g.
# test/combo posts) are left null — they have no place to inherit.
DA_NANG_DESTINATION_ID = "nwzsg28kmk8h4xkai5udjwy7"  # Da Nang (temporary fallback)
final = final.withColumn(
    "destination_id",
    F.when(F.col("business_service_id").isNotNull() & F.col("destination_id").isNull(),
           F.lit(DA_NANG_DESTINATION_ID))
     .otherwise(F.col("destination_id")))

# --- EXACTLY the 25 columns of product_field_mapping.md §A (fail loud if any missing) ---
ORDERED = [
    "product_id", "name", "lang", "product_type",
    "is_active", "is_bookable", "product_line", "price_level",
    "original_price", "marketing_price", "markup_percent",
    "variant_count", "active_variant_count", "variants_json",
    "variant_tickets", "variant_dates", "variant_hotels", "variant_age_groups",
    "variant_room_types", "variant_departures", "variant_statuses",
    "product_attributes","source_id", "destination_id",
    "updated_at",
]
final = final.select(*ORDERED)

# Drop products with no POI linkage: the KB needs BOTH source_id (AT_POI link)
# and destination_id (destination filter). A row missing either is unusable for
# hydration/filtering, so exclude it from the gold table.
final = final.where(F.col("source_id").isNotNull() & F.col("destination_id").isNotNull())

# Validation: source_id MUST reference a POI that exists in the enriched POI table
# (ai_enriched_poi = the POI set actually ingested as KB POI nodes). A product whose
# source_id is absent there cannot resolve its product->POI link in the KB
# (UnresolvedProductPOILinkError), so drop it. NOTE: ai_enriched_poi can be a SUBSET
# of ai_poi, so a source_id present in ai_poi may still be missing here.
_enriched_src = (spark.read.table(ENRICHED_POI_TABLE)
                     .select(F.col("source_id").cast("string").alias("_es_src"))
                     .where(F.col("_es_src").isNotNull())
                     .distinct())
_before = final.count()

# --- Surface the FAILED POIs: products whose source_id is NOT in ai_enriched_poi.
# These are dropped below (their POI was never ingested as a KB node, so the
# product->POI link can't resolve). Print them instead of dropping silently.
_failed = (final.select("product_id", "name", "product_type", "source_id", "destination_id")
                .join(_diag.select("product_id", "business_service_id"), "product_id", "left")
                .join(_enriched_src,
                      F.col("source_id").cast("string") == F.col("_es_src"), "left_anti"))
_n_failed = _failed.count()
if _n_failed:
    print(f"enriched-POI validation: {_n_failed} products reference a POI ABSENT from "
          f"{ENRICHED_POI_TABLE} (POI not enriched) -> will be DROPPED:")
    (_failed.orderBy("product_type", "name").show(min(_n_failed, 200), truncate=False))
    print(f"  -> {_failed.select('source_id').distinct().count()} distinct un-enriched POI source_id; "
          f"top by products dropped:")
    (_failed.groupBy("source_id", "destination_id", "business_service_id")
            .agg(F.count("*").alias("products_dropped"))
            .orderBy(F.desc("products_dropped"))
            .show(50, truncate=False))
else:
    print(f"enriched-POI validation: every product's source_id exists in {ENRICHED_POI_TABLE} (none dropped).")

if TEST_MODE:
    print(f"enriched-POI validation: TEST_MODE=True, skipping drop "
          f"({_n_failed} products would have been dropped in prod)")
else:
    final = final.join(F.broadcast(_enriched_src),
                       final["source_id"].cast("string") == F.col("_es_src"), "left_semi")
    print(f"enriched-POI validation: kept {final.count()}/{_before} products whose "
          f"source_id exists in {ENRICHED_POI_TABLE}")
print(f"enriched-POI validation: kept {final.count()}/{_before} products whose "
      f"source_id exists in {ENRICHED_POI_TABLE}")

target = TARGET_TABLE if not TARGET_LH else f"{TARGET_LH}.{TARGET_TABLE}"
(final.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(target))
print(f"wrote {final.count()} product rows ({len(ORDERED)} cols) -> {target}")

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 18, Finished, Available, Finished, False)

POI linkage: source_id <- business_service_id; 20/60 products match an ai_poi row for destination_id (10 distinct source_id)
enriched-POI validation: 40 products reference a POI ABSENT from lh_vv_gold.dbo.ai_enriched_poi (POI not enriched) -> will be DROPPED:
+------------------------------------+-------------------------------------------------------------------+------------+------------------------------------+------------------------+------------------------------------+
|product_id                          |name                                                               |product_type|source_id                           |destination_id          |business_service_id                 |
+------------------------------------+-------------------------------------------------------------------+------------+------------------------------------+------------------------+------------------------------------+
|be387a92-a155-47d0-82df-3d2e79da33ad|Ba Na Dining Voucher                         

In [20]:
%%sql
select * from lh_vv_gold.dbo.ai_product

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 22, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 25 fields>

quick fix, inactve old product

In [32]:
%%sql
select * from lh_vv_gold.dbo.ai_product

StatementMeta(, c2cb5f93-21db-4e27-bf14-ad0efb67e9f8, 34, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 25 fields>